# Notebook 02 — Embedding & Vector Store Exploration
Verify embedding quality, inspect Qdrant collection stats, and test retrieval accuracy.

In [ ]:
import sys
sys.path.insert(0, '../src')

from config import QDRANT_PATH, QDRANT_COLLECTION, EMBEDDING_MODEL_ID, EMBEDDING_DIM
print('Qdrant path       :', QDRANT_PATH)
print('Collection        :', QDRANT_COLLECTION)
print('Embedding model   :', EMBEDDING_MODEL_ID)
print('Embedding dim     :', EMBEDDING_DIM)

## 1. Check GPU availability

In [ ]:
import torch
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
    print('VRAM total     :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')

## 2. Load embedding model and embed sample sentences

In [ ]:
from vector_store import get_embeddings

embeddings = get_embeddings()

test_sentences = [
    'Total revenue for fiscal year 2023 was $383 billion.',
    'Net income attributable to Apple Inc. was $97 billion.',
    'The company faces risks from macroeconomic conditions.',
    'Debt-to-equity ratio calculation from balance sheet.',
]

vectors = embeddings.embed_documents(test_sentences)
print(f'Embedding shape: {len(vectors)} x {len(vectors[0])}')
print(f'Expected dim   : {EMBEDDING_DIM}')
assert len(vectors[0]) == EMBEDDING_DIM, 'Dimension mismatch!'

## 3. Cosine similarity between embeddings

In [ ]:
import numpy as np
import plotly.express as px
import pandas as pd

vecs = np.array(vectors)
# Vectors are already normalized (normalize_embeddings=True)
sim_matrix = vecs @ vecs.T

labels = [s[:40] + '...' for s in test_sentences]
sim_df  = pd.DataFrame(sim_matrix, index=labels, columns=labels)

fig = px.imshow(
    sim_df, text_auto='.2f',
    title='Cosine Similarity Matrix (BGE-large-en-v1.5)',
    color_continuous_scale='Blues',
    zmin=0, zmax=1
)
fig.show()

## 4. Qdrant collection stats

In [ ]:
from vector_store import get_qdrant_client
from config import QDRANT_COLLECTION

client = get_qdrant_client()
collections = client.get_collections().collections
print(f'Collections: {[c.name for c in collections]}')

if QDRANT_COLLECTION in [c.name for c in collections]:
    info = client.get_collection(QDRANT_COLLECTION)
    print(f'\nCollection: {QDRANT_COLLECTION}')
    print(f'  Vectors count : {info.vectors_count}')
    print(f'  Points count  : {info.points_count}')
    print(f'  Status        : {info.status}')
else:
    print(f'Collection "{QDRANT_COLLECTION}" not found. Run ingestion + indexing first.')

## 5. Retrieval smoke test — text query

In [ ]:
from vector_store import load_vector_store, similarity_search

vs = load_vector_store()

query = 'Apple total revenue fiscal 2023'
results = similarity_search(query, vs, k=5, filter_ticker='AAPL', filter_year=2023)

print(f'Query: {query}')
print(f'Results: {len(results)}\n')
for i, doc in enumerate(results, 1):
    m = doc.metadata
    print(f'[{i}] {m["company_ticker"]} | {m["filing_type"]} | {m["year"]} | {m["chunk_type"]}')
    print(f'    {doc.page_content[:200]}\n')

## 6. Retrieval smoke test — prefer_tables

In [ ]:
query = 'debt to equity ratio balance sheet'
results_table = similarity_search(query, vs, k=5, prefer_tables=True)
results_plain = similarity_search(query, vs, k=5, prefer_tables=False)

print('With prefer_tables=True:')
for r in results_table:
    print(f'  [{r.metadata["chunk_type"]:5s}] {r.page_content[:100]}')

print('\nWith prefer_tables=False:')
for r in results_plain:
    print(f'  [{r.metadata["chunk_type"]:5s}] {r.page_content[:100]}')

## 7. VRAM usage after embedding model load

In [ ]:
import torch
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / 1e9
    reserved  = torch.cuda.memory_reserved(0)  / 1e9
    total     = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Allocated : {allocated:.2f} GB')
    print(f'Reserved  : {reserved:.2f} GB')
    print(f'Total     : {total:.2f} GB')
    print(f'Free      : {total - reserved:.2f} GB')